# Day 27 — Python with MongoDB

**How to use this notebook:**
- Read each exercise, fill in your solution (replace `pass`)
- Run the test cell right after to see **APPROVED** or **FAILED**
- Do NOT modify the test cells

**Key concepts today:** MongoDB CRUD, `insert_one`, `find`, `update_one`, `delete_one`, `$set`, query operators

> MongoDB exercises use a **Python dict as an in-memory database** so no MongoDB installation is needed. The patterns are identical — only the driver calls differ.

---
## Exercise 1 — Insert Documents

Implement two insert functions on a collection (a Python list).

- `insert_one(collection, document)` — appends document, auto-assigns `'_id'` starting from 1. Returns the inserted document.
- `insert_many(collection, documents)` — inserts a list of documents. Returns count of inserted docs.

In [2]:
def insert_one(collection, document):
    document['_id'] = len(collection) + 1
    collection.append(document)
    return document

def insert_many(collection, documents):
    for documents in collection:
        documents['_id'] = len(collection) + 1
        collection.append(documents)
    return documents
        


In [3]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 1: insert ──')
col = []
doc = insert_one(col, {'name':'Alice','age':25})
_check('insert_one id',   doc.get('_id'), 1)
_check('insert_one name', doc.get('name'), 'Alice')
_check('collection size', len(col), 1)

count = insert_many(col, [{'name':'Bob','age':30},{'name':'Carol','age':28}])
_check('insert_many count', count, 2)
_check('ids assigned',      col[1]['_id'], 2)
_check('total size',        len(col), 3)


── Exercise 1: insert ──
  ✔ APPROVED — insert_one id
  ✔ APPROVED — insert_one name
  ✔ APPROVED — collection size


KeyboardInterrupt: 

---
## Exercise 2 — Query the Collection

Implement three query functions on a collection (a Python list acting as MongoDB).

- `find_by_field(collection, field, value)` — return a **list** of all documents
  where `document[field] == value`.
- `find_gt(collection, field, value)` — return a list of documents where
  `document[field] > value` (mirrors MongoDB `$gt`).
- `find_in(collection, field, values)` — return a list of documents where
  `document[field]` is **in** the iterable `values` (mirrors MongoDB `$in`).

In [4]:
def find_by_field(collection, field, value):
    return [i for i in collection if i[field] == value]

def find_gt(collection, field, value):
    return [i for i in collection if i[field] > value]

def find_in(collection, field, values):
    return [i for i in collection if i[field] in values]

In [5]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 2: query functions ──')
col = [
    {'_id': 1, 'name': 'Alice', 'age': 25, 'dept': 'eng'},
    {'_id': 2, 'name': 'Bob',   'age': 30, 'dept': 'hr'},
    {'_id': 3, 'name': 'Carol', 'age': 22, 'dept': 'eng'},
    {'_id': 4, 'name': 'Dave',  'age': 35, 'dept': 'eng'},
]

_check('find_by_field eng count',
       len(find_by_field(col, 'dept', 'eng')), 3)
_check('find_by_field hr count',
       len(find_by_field(col, 'dept', 'hr')),  1)
_check('find_by_field no match',
       find_by_field(col, 'dept', 'finance'),  [])

_check('find_gt age>25 count',
       len(find_gt(col, 'age', 25)), 2)
_check('find_gt age>35 count',
       len(find_gt(col, 'age', 35)), 0)

result_in = find_in(col, 'name', ['Alice', 'Carol'])
_check('find_in count',       len(result_in), 2)
_check('find_in names sorted',
       sorted(d['name'] for d in result_in), ['Alice', 'Carol'])
_check('find_in no match',
       find_in(col, 'name', ['Zara']), [])


── Exercise 2: query functions ──
  ✔ APPROVED — find_by_field eng count
  ✔ APPROVED — find_by_field hr count
  ✔ APPROVED — find_by_field no match
  ✔ APPROVED — find_gt age>25 count
  ✔ APPROVED — find_gt age>35 count
  ✔ APPROVED — find_in count
  ✔ APPROVED — find_in names sorted
  ✔ APPROVED — find_in no match


---
## Mini-Project — Student Database

Build a complete student database using the CRUD functions above (you may reuse them).

- `StudentDB` class with an internal `collection` list
- `add(name, age, grades)` — insert student, return student dict
- `get(name)` — find by name, return student or `None`
- `update_grade(name, subject, grade)` — update one grade in the grades dict, return `True/False`
- `remove(name)` — delete student, return `True/False`
- `top_students(n)` — return names of top `n` students sorted by average grade (highest first)

In [12]:
class StudentDB:
    def __init__(self):
        self.collection = []

    def add(self, name, age, grades):
        student_dict = {'name': name, 'age': age, 'grades': grades}
        self.collection.append(student_dict)
        return student_dict

    def get(self, name):
        for post in self.collection:
            if post['name'] == name:
                return post
            
        return None

    def update_grade(self, name, subject, grade):
        student_name = self.get(name)
        if student_name is None:
            return False
        else:
            student_name['grades'][subject] = grade
            return True

    def remove(self, name):
        for student in self.collection:
            if student['name'] == name:
                self.collection.remove(student)
                return True
        return False

    def top_students(self, n):
        top_stu_list = sorted(self.collection, key=lambda s: sum(student['grades'].values()) / len(student['grades'].values()) , reverse=True) 
        return [s['name'] for s in top_stu_list[:n]]

        

    

In [13]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Mini-Project: StudentDB ──')
db = StudentDB()
db.add('Alice', 20, {'math':90,'science':85,'english':92})
db.add('Bob',   21, {'math':60,'science':70,'english':65})
db.add('Carol', 22, {'math':78,'science':80,'english':75})

_check('get Alice',      db.get('Alice').get('name'),  'Alice')
_check('get missing',    db.get('Eve'),                None)

_check('update grade',   db.update_grade('Bob','math',75),  True)
_check('bob math after', db.get('Bob')['grades']['math'],   75)
_check('update missing', db.update_grade('Eve','math',90),  False)

_check('top 2',   db.top_students(2), ['Alice','Carol'])

_check('remove Alice',  db.remove('Alice'), True)
_check('remove missing',db.remove('Eve'),   False)
_check('size after',    len(db.collection), 2)


── Mini-Project: StudentDB ──
  ✔ APPROVED — get Alice
  ✔ APPROVED — get missing
  ✔ APPROVED — update grade
  ✔ APPROVED — bob math after
  ✔ APPROVED — update missing


NameError: name 'student' is not defined